In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import numpy as np
from torchvision.transforms.v2 import RandAugment
import torch
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn
import yaml
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../models"))

from dataset import get_data_loaders
from backbone import BackBone
from multiheadmodel import MultiHeadModel
from utils import deterministic, train

### Plots

In [ ]:
path = "../results/backbone_history.yaml"
with open(path, "r") as f:
    backbone_history = yaml.safe_load(f)

fontsize = 14
fig = plt.figure(figsize=(14, 6))
plt.plot(backbone_history["loss"], label="train support acc")
plt.xlabel("Epoch", fontsize=fontsize)
plt.ylabel("Loss", fontsize=fontsize)
plt.title("Backbone Training Loss", fontsize=fontsize)
plt.legend(fontsize=fontsize)
plt.grid(True)

fig.tight_layout()
path = "../images/backbone_training.png"
os.makedirs(os.path.dirname(path), exist_ok=True)
plt.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
tsne_path = "../results/backbone_tsne.npz"
with np.load(tsne_path) as data:
    tsne_embeddings = data["embeddings"]
    tsne_labels = data["labels"]
    snapshot_epochs = data["epochs"]

class_ticks = np.unique(tsne_labels)

fig, axes = plt.subplots(1, tsne_embeddings.shape[0], figsize=(12, 5), sharey=True)
if tsne_embeddings.shape[0] == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    scatter = ax.scatter(
        tsne_embeddings[i, :, 0],
        tsne_embeddings[i, :, 1],
        c=tsne_labels[i],
        cmap="tab10",
        s=10,
        alpha=0.8,
        vmin=class_ticks.min(),
        vmax=class_ticks.max(),
    )
    ax.set_title(f"Epoch {int(snapshot_epochs[i])}")
    ax.set_xlabel("t-SNE dim 1", fontsize=14)
    if i == 0:
        ax.set_ylabel("t-SNE dim 2", fontsize=14)
    if i == len(axes) - 1:
        cbar = fig.colorbar(scatter, ax=ax, ticks=class_ticks, label="Class")
        cbar.ax.set_yticklabels([str(int(cls)) for cls in class_ticks])
    ax.grid(True)

fig.tight_layout()
path = "../images/backbone_training_tsnes.png"
os.makedirs(os.path.dirname(path), exist_ok=True)
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

### Training with classifier

In [3]:
backbone= BackBone()
backbone.load_state_dict(torch.load("../models/weights/backbone.pth"))
model = MultiHeadModel(backbone)
model.add_head(0, 2)

c:\Users\User\miniconda3\envs\vision\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\User\miniconda3\envs\vision\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
C:\Users\User\AppData\Local\Temp\ipykernel_10984\1640064543.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weigh

In [4]:
for param in model.backbone.parameters(): 
    param.requires_grad = False

In [5]:
dataloaders = get_data_loaders(batch_size=512)
task0_train = dataloaders[0][0]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

/home/alumno1/miniconda3/envs/vision/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Task [0, 1]: Train=9000, Val=1000, Test=2000
Task [2, 3]: Train=9000, Val=1000, Test=2000
Task [4, 5]: Train=9000, Val=1000, Test=2000
Task [6, 7]: Train=9000, Val=1000, Test=2000
Task [8, 9]: Train=9000, Val=1000, Test=2000


MultiHeadModel(
  (backbone): BackBone(
    (encoder): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): Identity()
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runn

In [ ]:
deterministic(42)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

epochs = 20

train(model, dataloaders[0][0], optimizer, criterion, "task0_4_2", epochs, 0)

Epochs: 100%|██████████| 20/20 [00:10<00:00,  1.96epoch/s, loss=0.00774]



In [7]:
task0_val = dataloaders[0][1]

model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for x, y in tqdm(task0_val, desc="Evaluating", unit="batch"):
        x, y = x.to(device), y.to(device)
        pred = model(x, 0)
        _, predicted = torch.max(pred.data, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()
    print(f"Accuracy: {100 * correct / total:.2f}%")

Evaluating: 100%|██████████| 2/2 [00:00<00:00, 11.43batch/s]

Accuracy: 99.40%
